In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from NEDAS import get_scheme

In [ ]:
scheme = get_scheme(config_file="vort2d/config.yml", debug=True , nproc=4)

In [ ]:
scheme()

In [ ]:
model = scheme.c.models['vort2d']
dataset = scheme.c.datasets['vort2d']

In [ ]:
model.memory.keys()

In [ ]:
scheme.c.time = scheme.c.config.time_start

truth_state = []
while scheme.c.time < scheme.c.config.time_end:
    truth_state.append(model.memory['truth']['state'][None, scheme.c.time])
    scheme.c.time = scheme.c.next_time


In [ ]:
scheme.c.time = scheme.c.config.time_start

prior_state = []
post_state = []
while scheme.c.time < scheme.c.config.time_end:
    ens = []
    for m in range(scheme.c.nens):
        ens.append(model.memory['prior']['state'][m, scheme.c.time])
    prior_state.append(ens)
    ens = []
    for m in range(scheme.c.nens):
        if scheme.c.time >= scheme.c.config.time_analysis_start and scheme.c.time <= scheme.c.config.time_analysis_end:
            ens.append(model.memory['post']['state'][m, scheme.c.time])
        else:
            ens.append(np.full(scheme.c.grid.x.shape, np.nan))
    post_state.append(ens)
    scheme.c.time = scheme.c.next_time


In [ ]:
nobs = scheme.c.obs.info.records[0].nobs
scheme.c.time = scheme.c.config.time_start

obs_val = []
obs_x = []
while scheme.c.time < scheme.c.config.time_end:
    if scheme.c.time >= scheme.c.config.time_analysis_start and scheme.c.time <= scheme.c.config.time_analysis_end:
        obs_val.append(dataset.memory['raw']['state'][None, scheme.c.time]['obs'])
        obs_x.append(dataset.memory['raw']['state'][None, scheme.c.time]['x'])
    else:
        obs_val.append(np.full(nobs, np.nan))
        obs_x.append(np.full(nobs, np.nan))
    scheme.c.time = scheme.c.next_time

In [ ]:
plt.contourf(np.array(truth_state).T)

In [ ]:
m = 1
plt.contourf((np.array(prior_state)[:, m, :] - np.array(truth_state)).T)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10, 3))

n = 50

# plot prior/post ensemble
for m in range(scheme.c.nens):
    ax.plot(scheme.c.grid.x, post_state[n][m], color='c')

# plot obs
ax.plot(obs_x[n], obs_val[n], 'x', color='r', markersize=10)

# plot truth
ax.plot(scheme.c.grid.x, truth_state[n], color='k', linewidth=2)

#ax.set_xlim(0, scheme.c.grid.Lx)
#ax.set_ylim(-2, 5)

In [ ]:
scheme.c.obs.info.records[0].asdict()

In [ ]:
scheme.c.config.inflation_def